# DC Motor Generation

This notebook is the DC motor equivalent of `TestPendulumGeneration.ipynb`.

Instead of generating an `n`-link pendulum with symbolic mechanics, we use the autonomous no-input DC motor dataset added in `datasets/dc_motor.py`.

State:

```text
x = [omega, current]
```

Default equations:

```text
d omega / dt = (Kt * current - b * omega - Tl) / J
d current / dt = (Va - R * current - Ke * omega) / L
```

By default `Va=0` and `Tl=0`, so the motor is an autonomous dissipative system with a stable equilibrium at the origin.


In [ ]:
%matplotlib inline
import glob
import matplotlib.pyplot as plt
import numpy as np
import torch

from datasets.dc_motor import DEFAULTS, build, dc_motor_gradient
from util import DynamicLoad, latest_file


## Parameters and right-hand side

The helper `dc_motor_gradient(params)` returns a vectorized function `f(X)` that maps states to derivatives.


In [ ]:
params = DEFAULTS.copy()
params.update({
    "Va": 0.0,
    "Tl": 0.0,
})

rhs = dc_motor_gradient(params)

print("DC motor parameters")
for k, v in params.items():
    print(f"{k:>12s}: {v}")

print("\nDerivative at the origin:", rhs(np.array([0.0, 0.0], dtype=np.float32)))


## Linear system matrix

Because this no-input DC motor model is linear, we can write it as:

```text
x_dot = A x
```

where `x = [omega, current]`.


In [ ]:
J, b = params["J"], params["b"]
Kt, Ke = params["Kt"], params["Ke"]
R, L = params["R"], params["L"]

A = np.array([
    [-b / J,   Kt / J],
    [-Ke / L, -R / L],
], dtype=np.float64)

eigvals = np.linalg.eigvals(A)

print("A =")
print(A)
print("\nEigenvalues:", eigvals)
print("Stable linear system:", np.all(np.real(eigvals) < 0))


## Generate train/test datasets

This calls the same dataset builder that `train.py` uses through `DynamicLoad("datasets")`.

The returned tensors are:

```text
X: sampled states
Y: true derivatives f(X)
```


In [ ]:
train_ds = build({"n": "5000", "nocache": True})
test_ds = build({"n": "1000", "test": True, "nocache": True})

X_train, Y_train = train_ds.tensors
X_test, Y_test = test_ds.tensors

print("train X:", tuple(X_train.shape), "train Y:", tuple(Y_train.shape))
print("test  X:", tuple(X_test.shape), "test  Y:", tuple(Y_test.shape))

print("\nFirst five training samples:")
for x, y in zip(X_train[:5].numpy(), Y_train[:5].numpy()):
    print(f"x={x}, f(x)={y}")


## Vector field

This shows the learned target field: for every point `(omega, current)`, the arrow is `f(x)`.


In [ ]:
omega_grid = np.linspace(-5.0, 5.0, 25)
current_grid = np.linspace(-5.0, 5.0, 25)
Omega, Current = np.meshgrid(omega_grid, current_grid)

grid_states = np.stack([Omega.ravel(), Current.ravel()], axis=1).astype(np.float32)
grid_derivatives = rhs(grid_states)

dOmega = grid_derivatives[:, 0].reshape(Omega.shape)
dCurrent = grid_derivatives[:, 1].reshape(Current.shape)

speed = np.sqrt(dOmega**2 + dCurrent**2)
dOmega_n = dOmega / np.maximum(speed, 1e-8)
dCurrent_n = dCurrent / np.maximum(speed, 1e-8)

plt.figure(figsize=(7, 6))
plt.quiver(Omega, Current, dOmega_n, dCurrent_n, speed)
plt.scatter([0], [0], marker="x", s=80)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("DC motor vector field: x_dot = f(x)")
plt.colorbar(label="||f(x)||")
plt.tight_layout()
plt.show()


## RK4 simulation

The pendulum notebook integrates physical equations over time. Here we do the same with a fourth-order Runge-Kutta step.


In [ ]:
def rk4_step(rhs, x, dt):
    k1 = rhs(x)
    k2 = rhs(x + 0.5 * dt * k1)
    k3 = rhs(x + 0.5 * dt * k2)
    k4 = rhs(x + dt * k3)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

def simulate(rhs, x0, steps=300, dt=0.01):
    x = np.asarray(x0, dtype=np.float32)
    trajectory = np.zeros((steps, *x.shape), dtype=np.float32)
    trajectory[0] = x
    for step in range(1, steps):
        x = rk4_step(rhs, x, dt).astype(np.float32)
        trajectory[step] = x
    return trajectory

initial_states = np.array([
    [ 4.0,  4.0],
    [ 4.0, -4.0],
    [-4.0,  4.0],
    [-4.0, -4.0],
    [ 2.0,  0.0],
    [ 0.0,  2.0],
], dtype=np.float32)

dt = 0.01
steps = 400
trajectories = simulate(rhs, initial_states, steps=steps, dt=dt)
time = np.arange(steps) * dt

print("trajectories shape:", trajectories.shape)


## State traces

Both `omega` and `current` should decay toward zero because the default system is no-input and dissipative.


In [ ]:
plt.figure(figsize=(8, 4))
for idx in range(trajectories.shape[1]):
    plt.plot(time, trajectories[:, idx, 0], label=f"path {idx}")
plt.xlabel("time [s]")
plt.ylabel("omega [rad/s]")
plt.title("Angular velocity trajectories")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
for idx in range(trajectories.shape[1]):
    plt.plot(time, trajectories[:, idx, 1], label=f"path {idx}")
plt.xlabel("time [s]")
plt.ylabel("current [A]")
plt.title("Current trajectories")
plt.tight_layout()
plt.show()


## Phase portrait

This is the DC motor analogue of plotting pendulum trajectories. The trajectories should spiral/curve into the origin depending on parameters.


In [ ]:
plt.figure(figsize=(6, 6))
for idx in range(trajectories.shape[1]):
    plt.plot(trajectories[:, idx, 0], trajectories[:, idx, 1], label=f"path {idx}")
    plt.scatter(trajectories[0, idx, 0], trajectories[0, idx, 1], marker="o")
plt.scatter([0], [0], marker="x", s=100)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("DC motor phase portrait")
plt.legend()
plt.tight_layout()
plt.show()


# Long-horizon solver vs trained model comparison

This section compares the exact DC motor solver trajectory against a trained neural dynamics model over a longer rollout horizon.

Run `bash train_dc_motor_stable` first, or change `MODEL_SPEC` and `WEIGHT_GLOB` to point to another trained checkpoint.


In [ ]:
MODEL_SPEC = "stabledynamics[latent_space_dim=2,a=0.001,projfn=NN-REHU,projfn_eps=0.01,smooth_v=0,hp=60,h=100,rehu=0.001]"
WEIGHT_GLOB = "experiments/dc-motor-stable/checkpoint_*.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    weight_path = latest_file(WEIGHT_GLOB)
    model_module = DynamicLoad("models")(MODEL_SPEC)
    model = model_module.model
    model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    MODEL_AVAILABLE = True
    print(f"Loaded model from {weight_path} on {DEVICE}")
except Exception as exc:
    MODEL_AVAILABLE = False
    model = None
    weight_path = None
    print("No trained checkpoint available yet.")
    print("Train one with: bash train_dc_motor_stable")
    print("Original error:", repr(exc))


In [ ]:
def rk4_model_step(model, x, dt, device=DEVICE):
    # stabledynamics needs gradients with respect to x, so do not use torch.no_grad().
    x = x.detach().requires_grad_(True)
    k1 = model(x)
    k2 = model((x + 0.5 * dt * k1).detach().requires_grad_(True))
    k3 = model((x + 0.5 * dt * k2).detach().requires_grad_(True))
    k4 = model((x + dt * k3).detach().requires_grad_(True))
    return (x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)).detach()

def simulate_model(model, x0, steps=2000, dt=0.01, device=DEVICE):
    x = torch.tensor(x0, dtype=torch.float32, device=device)
    trajectory = np.zeros((steps, *x0.shape), dtype=np.float32)
    trajectory[0] = x.detach().cpu().numpy()
    for step in range(1, steps):
        x = rk4_model_step(model, x, dt, device=device)
        trajectory[step] = x.detach().cpu().numpy().astype(np.float32)
    return trajectory

def rollout_comparison(rhs, model, x0, steps=2000, dt=0.01):
    solver_traj = simulate(rhs, x0, steps=steps, dt=dt)
    model_traj = simulate_model(model, x0, steps=steps, dt=dt)
    error = solver_traj - model_traj
    return solver_traj, model_traj, error


In [ ]:
if MODEL_AVAILABLE:
    long_x0 = np.array([
        [ 5.0,  5.0],
        [ 5.0, -5.0],
        [-5.0,  5.0],
        [-5.0, -5.0],
        [ 3.0,  0.0],
        [ 0.0,  3.0],
        [ 1.0, -4.0],
        [-2.0,  4.0],
    ], dtype=np.float32)

    long_steps = 2000
    long_dt = 0.01
    long_time = np.arange(long_steps) * long_dt

    solver_long, model_long, long_error = rollout_comparison(rhs, model, long_x0, steps=long_steps, dt=long_dt)
    mean_norm_error = np.mean(np.linalg.norm(long_error, axis=2), axis=1)
    state_mse = np.mean(long_error**2, axis=(1, 2))

    print("Long-horizon comparison complete")
    print("horizon seconds:", long_steps * long_dt)
    print("final mean norm error:", mean_norm_error[-1])
    print("mean rollout MSE:", np.mean(state_mse))
else:
    print("Skipping long-horizon comparison until a checkpoint exists.")


## Long-horizon state traces

Solid lines are the numerical solver. Dashed lines are the trained model rollout.


In [ ]:
if MODEL_AVAILABLE:
    plt.figure(figsize=(10, 4))
    for idx in range(long_x0.shape[0]):
        plt.plot(long_time, solver_long[:, idx, 0], linestyle="-", linewidth=1.0)
        plt.plot(long_time, model_long[:, idx, 0], linestyle="--", linewidth=1.0)
    plt.xlabel("time [s]")
    plt.ylabel("omega [rad/s]")
    plt.title("Long-horizon omega: solver solid vs model dashed")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    for idx in range(long_x0.shape[0]):
        plt.plot(long_time, solver_long[:, idx, 1], linestyle="-", linewidth=1.0)
        plt.plot(long_time, model_long[:, idx, 1], linestyle="--", linewidth=1.0)
    plt.xlabel("time [s]")
    plt.ylabel("current [A]")
    plt.title("Long-horizon current: solver solid vs model dashed")
    plt.tight_layout()
    plt.show()


## Long-horizon phase portrait

This view is usually the easiest way to see whether the learned model has the same qualitative behavior as the physical solver.


In [ ]:
if MODEL_AVAILABLE:
    plt.figure(figsize=(7, 7))
    for idx in range(long_x0.shape[0]):
        plt.plot(solver_long[:, idx, 0], solver_long[:, idx, 1], linestyle="-", linewidth=1.0)
        plt.plot(model_long[:, idx, 0], model_long[:, idx, 1], linestyle="--", linewidth=1.0)
        plt.scatter(long_x0[idx, 0], long_x0[idx, 1], marker="o", s=20)
    plt.scatter([0], [0], marker="x", s=120)
    plt.xlabel("omega [rad/s]")
    plt.ylabel("current [A]")
    plt.title("Long-horizon phase portrait: solver solid vs model dashed")
    plt.tight_layout()
    plt.show()


## Long-horizon rollout error

This plot shows whether small one-step errors accumulate or decay over a long autonomous rollout.


In [ ]:
if MODEL_AVAILABLE:
    plt.figure(figsize=(8, 4))
    plt.semilogy(long_time, mean_norm_error, label="mean ||solver - model||")
    plt.semilogy(long_time, state_mse, label="state MSE")
    plt.xlabel("time [s]")
    plt.ylabel("error")
    plt.title("Long-horizon rollout error")
    plt.legend()
    plt.tight_layout()
    plt.show()
